# GeoSR-4 — SwinIR baseline training (Colab GPU)

Phase 4 (PRD section 27-30). Same workflow as the EDSR notebook: smoke-tested locally on CPU first (window-attention shape/gradient checks, see `decisions.md` D012), real training happens here.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision

## 2. Download the dataset (cross-sensor split only, ~2.1 GB)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Train
Default config: embed_dim=60, 4 RSTB blocks x depth 2, 6 heads, window_size=11 (matches the 121x121 LR patch size exactly -- see D012).

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 20 \
  --batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --checkpoint-dir experiments/swinir \
  --log-every 20

## 4. Full validation-set evaluation (fair comparison vs Bicubic + EDSR)
The per-epoch numbers above only sample 50 val pairs for speed -- this is the real number for decisions.md.

In [ ]:
!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir/swinir_epoch19.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Quality run: VGG perceptual loss + ICNR upsample fix
D023: plain L1 loss (step 3 above) tends to produce blurry output -- well documented in the SR literature. This run adds a VGG perceptual loss (pushes toward sharper, more realistic-looking output) and uses the ICNR-initialized upsample block (fixes the checkerboard/ripple artifact from D020). Same architecture as step 3, so this is a direct visual/quality comparison against the plain-L1 checkpoint.

In [ ]:
!python ml/training/train_swinir.py \
  --epochs 30 \
  --batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --lambda-perceptual 0.01 \
  --checkpoint-dir experiments/swinir_quality \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_quality/swinir_epoch29.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 6. Download the checkpoint
(Drive mount can be flaky -- direct browser download is simpler and doesn't need Google auth.)

In [ ]:
from google.colab import files
files.download('experiments/swinir_quality/swinir_epoch29.pt')
# uncomment if you also want the plain-L1 checkpoint from step 3/4:
# files.download('experiments/swinir/swinir_epoch19.pt')